# CIFAR-10 Image Classification

This is a **standalone notebook** — run it directly without needing the `.py` files.
All model definitions and training logic are inlined below.

autoresearch uses the `.py` scripts (`model.py`, `train.py`); this notebook is for
interactive exploration and prototyping.

In [ ]:
import json
import os
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

## Model Definition

ResNet-18 adapted for 32×32 CIFAR images (3×3 initial conv, no maxpool).

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes * self.expansion, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * self.expansion),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNet(nn.Module):
    """ResNet-18 for CIFAR-10 (32x32 input, no initial maxpool)."""

    def __init__(self, num_classes=10):
        super().__init__()
        self.in_planes = 64

        # CIFAR-specific: 3x3 conv instead of 7x7, no maxpool
        self.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)

        self.fc = nn.Linear(512 * BasicBlock.expansion, num_classes)

    def _make_layer(self, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(BasicBlock(self.in_planes, planes, s))
            self.in_planes = planes * BasicBlock.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.adaptive_avg_pool2d(out, (1, 1))
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out


def make_model(num_classes=10):
    return ResNet(num_classes=num_classes)

## Config & Device

In [ ]:
TIME_BUDGET = int(os.environ.get("AUTORESEARCH_TIME_BUDGET", 120))
BATCH_SIZE = 128
LR = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
NUM_WORKERS = 2
DATA_DIR = os.path.join("data", "cifar-10")

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

## Load Data

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

testset = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train: {len(trainset)} samples, Val: {len(testset)} samples")

## Train

In [ ]:
model = make_model(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params / 1e6:.2f}M")

t_start = time.time()
epoch = 0
best_val_acc = 0.0

while True:
    elapsed = time.time() - t_start
    if elapsed >= TIME_BUDGET:
        break

    epoch += 1
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in trainloader:
        if time.time() - t_start >= TIME_BUDGET:
            break
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    scheduler.step()
    train_acc = correct / total if total > 0 else 0.0

    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            val_total += targets.size(0)
            val_correct += predicted.eq(targets).sum().item()

    val_acc = val_correct / val_total
    best_val_acc = max(best_val_acc, val_acc)
    print(f"Epoch {epoch}: train_acc={train_acc:.4f} val_acc={val_acc:.4f} lr={scheduler.get_last_lr()[0]:.6f}")

## Results

In [ ]:
total_time = time.time() - t_start

if device.type == "cuda":
    peak_memory_mb = torch.cuda.max_memory_allocated() / 1024 / 1024
elif device.type == "mps":
    peak_memory_mb = torch.mps.driver_allocated_size() / 1024 / 1024
else:
    peak_memory_mb = 0.0

metrics = {
    "val_accuracy": float(best_val_acc),
    "training_seconds": round(total_time, 1),
    "peak_memory_mb": round(peak_memory_mb, 1),
    "num_epochs": epoch,
    "num_params_M": round(num_params / 1e6, 2),
}

with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Best val_accuracy: {best_val_acc:.6f}")
print(f"Training time: {total_time:.1f}s")
print(f"Peak memory: {peak_memory_mb:.1f} MB")
print(f"Metrics written to metrics.json")